In [1]:
!pip install pycountry

In [2]:
import wbdata
import pandas as pd
import datetime
import os
import pycountry

Key '7358584228840476562' not in persistent cache.
Key '3584482554026485528' not in persistent cache.
Key '2619766637911086107' not in persistent cache.
Key '57522779939512972' not in persistent cache.
Key '6592664866940242715' not in persistent cache.
Key '-4644718436267654806' not in persistent cache.
Key '-9011469416302398483' not in persistent cache.
Key '-635674806335491026' not in persistent cache.
Key '-4409776481149482060' not in persistent cache.
Key '1671536891369975344' not in persistent cache.
Key '-890706966601597791' not in persistent cache.
Key '-1401594168235724873' not in persistent cache.
Key '5807766602265929222' not in persistent cache.
Key '1441015622231185545' not in persistent cache.
Key '-30614650436029551' not in persistent cache.
Key '-3804716325250020585' not in persistent cache.
Key '-4769432241365420006' not in persistent cache.
Key '-285646810702136316' not in persistent cache.
Key '-796534012336263398' not in persistent cache.
Key '2014469375181022569' no

In [3]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
PROCESSED_DATA_DIR_PATH = os.path.join(DATA_DIR_PATH, "processed_data")

In [4]:
# 1. Define the indicators you want to fetch
indicators = {
    "SP.POP.GROW": "pop_growth",
    "SP.POP.TOTL": "pop_total",
    "SP.URB.TOTL.IN.ZS": "pop_urban_pct",
    "SI.POV.DDAY": "poverty_headcount_ratio",
    "EG.ELC.ACCS.ZS": "access_to_electricity_pct",
    "EG.USE.ELEC.KH.PC": "electricity_consumption_per_capita_kwh",
    "EG.USE.PCAP.KG.OE": "energy_use_per_capita_kg_of_oil_equivalent",
    "EG.USE.COMM.GD.PP.KD": "energy_use_kg_of_oil_equivalent_per_gdp",
    "EG.ELC.COAL.ZS": "electricity_from_coal_pct",
    "EG.ELC.NGAS.ZS": "electricity_from_natural_gas_pct",
    "EG.ELC.RNWX.ZS": "electricity_from_renewables_pct",
    "EG.FEC.RNEW.ZS": "renewable_energy_consumption_pct",
    "AG.LND.FRST.ZS": "forest_area_pct",
    "NV.AGR.TOTL.ZS": "agriculture_value_added_pct",
    "ER.H2O.FWTL.ZS": "annual_freshwater_withdrawals_pct"
}

date_range = (datetime.datetime(2000,1,1), datetime.datetime(2022,1,1))

df = wbdata.get_dataframe(
    indicators,
    country="all",
    date=date_range,
    freq='Y',
    parse_dates=True
).reset_index().rename(columns={'country':'region','date':'year'})

# 2) Pull the country lookup (wbdata.get_countries returns a list of dicts)
country_list = wbdata.get_countries()  

#  Inspect one entry to see its keys:
# print(country_list[0])
# -> {'id': 'ABW', 'iso2Code': 'AW', 'name': 'Aruba', …}

# 3) Build a small DataFrame of name → ISO-2
country_df = pd.DataFrame(country_list)[['name','iso2Code']]
country_df.columns = ['region','iso2']  

# 4) Merge it onto your indicators DataFrame
df = df.merge(country_df, on='region', how='left')

# 5) (Optional) Convert ISO-2 → ISO-3 with pycountry
def iso2_to_iso3(c2):
    try:
        return pycountry.countries.get(alpha_2=c2).alpha_3
    except:
        return None

df['iso_alpha_3'] = df['iso2'].map(iso2_to_iso3)

# make year only the year
df['year'] = df['year'].dt.year

# drop iso2 and region
df = df.drop(columns=['iso2', 'region'])


In [5]:
df

,year,pop_growth,pop_total,pop_urban_pct,poverty_headcount_ratio,access_to_electricity_pct,electricity_consumption_per_capita_kwh,energy_use_per_capita_kg_of_oil_equivalent,energy_use_kg_of_oil_equivalent_per_gdp,electricity_from_coal_pct,electricity_from_natural_gas_pct,electricity_from_renewables_pct,renewable_energy_consumption_pct,forest_area_pct,agriculture_value_added_pct,annual_freshwater_withdrawals_pct,iso_alpha_3
0,2022,2.626659,731821393.0,37.909012,NaN,48.801258,497.161668,565.488909,136.911837,53.452960,3.020806,NaN,NaN,29.737205,13.840745,NaN,None
1,2021,2.684849,713090928.0,37.393633,NaN,48.127211,513.745842,570.998888,139.730030,54.663797,2.615079,4.939318,NaN,29.955194,13.404438,5.041739,None
2,2020,2.736283,694446100.0,36.884034,NaN,46.282371,512.766661,563.976201,140.687987,56.721763,2.449528,4.585792,66.123449,30.174252,14.644268,5.012011,None
3,2019,2.759057,675950189.0,36.384272,NaN,44.390861,548.496602,586.441491,138.644201,58.600950,2.731900,5.173796,63.387090,30.391626,12.627574,4.960944,None
4,2018,2.771987,657801085.0,35.893398,NaN,43.035073,568.141299,583.763039,137.485602,60.485772,2.035550,3.945917,62.242631,30.611512,11.913986,4.917538,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6113,2004,1.086049,12365896.0,34.294000,NaN,35.600000,831.803858,449.272022,158.818150,43.183455,0.000000,0.893713,81.800000,46.999354,18.063797,32.226754,ZWE
6114,2003,1.189736,12232323.0,34.479000,NaN,35.100000,861.651544,477.200797,157.177877,39.095352,0.000000,0.595068,78.000000,47.118444,14.793355,33.262643,ZWE
6115,2002,0.962220,12087653.0,34.585000,NaN,34.200000,868.654982,519.581800,140.372196,55.467567,0.000000,0.815768,74.700000,47.237534,12.568368,34.298532,ZWE
6116,2001,0.669179,11971901.0,34.170000,NaN,34.200000,868.533744,540.270614,131.706288,62.079433,0.000000,1.686061,72.300000,47.356624,15.627071,32.675367,ZWE


In [6]:
df.to_csv(os.path.join(PROCESSED_DATA_DIR_PATH, 'wb_control_vars.csv'), index=False)